# X-REACH RL Retraining — Data-Augmented DQN & PPO (Google Colab)

This notebook retrains the two RL prediction models (`DQN`, `PPO`) used by
X-REACH's backend (`backend/services/modelService.js`), starting from the same
raw dataset (`heart_disease_trimmed.csv`, 539 rows) and the same network
architectures/environments as the original research notebooks
(`research/reinforcement_learning/EHR_DQN_PyTorch.ipynb`, `PPO.ipynb`), with
targeted fixes for problems diagnosed against the currently deployed
artifacts:

- **PPO's confidence always showing exactly 0% or 100%.** Traced to its
  actor head's output logits growing unboundedly during training because the
  entropy bonus that's supposed to resist that collapse (`ENTROPY_COEFFICIENT
  = 0.01`, constant, never annealed) was too weak relative to how many
  gradient updates 500 episodes × ~10 PPO epochs produces.
- **Non-monotonic, sometimes contradictory behavior** (e.g. `age` scoring a
  20-year-old as higher risk than a 65-year-old; two risk factors together
  sometimes flipping a HIGH prediction back to LOW). Root cause: only 539
  real training rows, reused verbatim every single episode with no
  diversity, is very easy for a nonlinear network to overfit/memorize noise
  on weaker-signal features (per the HRFLM global importance report,
  `age` ranks 8th of 10 features).

## What this notebook changes vs. the original research notebooks

1. **Clinically-safe data augmentation**, applied *only* to each fold's
   training split, never its test split (see Section 5 for why that
   boundary matters for a medical dataset).
2. Reuses the **exact, already-deployed RobustScaler parameters** from
   `models/rl/preprocessing_params.json` (not refit on augmented data) so
   the retrained model stays compatible with the existing backend with
   **zero backend code changes** and no database reseeding.
3. **PPO**: entropy coefficient raised from `0.01` to `0.05` (still constant
   throughout training, just stronger) + dropout added to the actor/critic
   heads, to resist saturating to exactly 0%/100%.
4. **DQN**: L2 weight decay + dropout added to reduce overfitting on
   weak-signal features, and each training episode now shuffles which real
   patient it starts from (the original always reset to the same first
   training row every episode).
5. **Export cells fixed** to (a) use the real trained state dimension (15),
   fixing a stale hardcoded `best_state_dimension = 10` bug found in the
   original PPO export cell, and (b) record `runtime_compatibility.output_activation`
   in `metadata.json` (`"softmax"` for PPO's actor output — which already
   applies its own softmax — `"none"` for DQN's raw Q-values). This field is
   **required** by the currently deployed `backend/services/adapters/onnxAdapter.js`;
   without it PPO's output gets softmax'd twice and collapses right back to
   0%/100%.
6. Adds an explicit **post-training sanity-check cell** (Section 10) that
   sweeps `age`, `oldpeak`, `thalach`, `trestbps` one at a time, the same
   kind of check used to diagnose the original problems — so you can see
   directly whether retraining actually fixed the erratic behavior *before*
   deploying it.

**How to use this notebook:** Runtime → Change runtime type → choose a GPU
(optional, CPU is fine for this dataset size) → Run all. When prompted,
upload `heart_disease_trimmed.csv` (from `research/data/` in the X-REACH
repo). At the end it triggers browser downloads for both models'
`model.pth`, `model.onnx`, and `metadata.json`.


## 1. Setup

In [ ]:
!pip install -q scikit-learn onnx --upgrade

import os
import json
import random
import shutil
from datetime import datetime, timezone
from collections import deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2. Upload the raw dataset

Upload `heart_disease_trimmed.csv` (found at `research/data/` in the X-REACH
repo — 539 rows, the 10 raw canonical features + `target`). This is the
*original, real* dataset; nothing synthetic goes in at this stage.


In [ ]:
CSV_PATH = "heart_disease_trimmed.csv"

if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print("Please upload heart_disease_trimmed.csv")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"{CSV_PATH} not found and google.colab is not available to "
            "prompt an upload. Place the file next to this notebook."
        )

raw_df = pd.read_csv(CSV_PATH)
print("Loaded", len(raw_df), "real patient rows")
print("Target distribution:\n", raw_df["target"].value_counts())
raw_df.head()


## 3. Canonical feature contract (must match the backend exactly)

This mirrors `backend/utils/featureContract.js` and
`backend/services/preprocessingService.js` byte-for-byte, using the
**already-fitted, already-deployed** scaler parameters from
`models/rl/preprocessing_params.json`. We deliberately do **not** refit the
scaler on the augmented data — reusing the exact fixed parameters is what
lets the retrained model drop into the existing backend with no other
changes.


In [ ]:
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs",
]

# Valid clinical ranges (backend/utils/featureContract.js FEATURE_VALIDATION)
# -- used as a hard safety clip on every synthetic row, so augmentation can
# never produce a physiologically impossible patient.
FEATURE_RANGES = {
    "thalach": (60, 250),
    "oldpeak": (0, 10),
    "age": (1, 120),
    "trestbps": (50, 250),
}
CONTINUOUS_FEATURES = ["age", "trestbps", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["sex", "fbs", "exang", "cp", "restecg", "slope"]

# Fixed RobustScaler params -- copied verbatim from
# models/rl/preprocessing_params.json. DO NOT refit these on augmented data.
SCALER_PARAMS = {
    "age":      {"median": 56.0,  "iqr": 12.0},
    "trestbps": {"median": 130.0, "iqr": 22.0},
    "thalach":  {"median": 140.0, "iqr": 38.5},
    "oldpeak":  {"median": 1.0,   "iqr": 1.9},
}

CP_INT_TO_ONEHOT = {
    0: "cp_typical angina",
    1: "cp_atypical angina",
    2: "cp_non-anginal",
    3: "cp_asymptomatic",
}
RESTECG_INT_TO_ONEHOT = {
    0: "restecg_normal",
    1: "restecg_st-t abnormality",
    2: "restecg_lv hypertrophy",
}

# Fixed 15-dim output order the deployed models were trained on -- see
# models/rl/preprocessing_params.json "trained_feature_order_15dim".
TRAINED_FEATURE_ORDER_15DIM = [
    "sex", "fbs", "exang", "age", "trestbps", "thalach", "oldpeak",
    "cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
    "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality",
    "slope",
]


def preprocess_row(row):
    """Raw canonical feature dict/Series -> 15-dim np.float32 vector.
    Mirrors backend/services/preprocessingService.js::toTrainedRepresentation
    exactly (same formula, same fixed scaler params, same one-hot layout).
    """
    vector = {
        "sex": row["sex"], "fbs": row["fbs"], "exang": row["exang"],
        "age": (row["age"] - SCALER_PARAMS["age"]["median"]) / SCALER_PARAMS["age"]["iqr"],
        "trestbps": (row["trestbps"] - SCALER_PARAMS["trestbps"]["median"]) / SCALER_PARAMS["trestbps"]["iqr"],
        "thalach": (row["thalach"] - SCALER_PARAMS["thalach"]["median"]) / SCALER_PARAMS["thalach"]["iqr"],
        "oldpeak": (row["oldpeak"] - SCALER_PARAMS["oldpeak"]["median"]) / SCALER_PARAMS["oldpeak"]["iqr"],
        "cp_asymptomatic": 0, "cp_atypical angina": 0, "cp_non-anginal": 0, "cp_typical angina": 0,
        "restecg_lv hypertrophy": 0, "restecg_normal": 0, "restecg_st-t abnormality": 0,
        "slope": row["slope"],
    }
    vector[CP_INT_TO_ONEHOT[int(row["cp"])]] = 1
    vector[RESTECG_INT_TO_ONEHOT[int(row["restecg"])]] = 1
    return np.array([vector[k] for k in TRAINED_FEATURE_ORDER_15DIM], dtype=np.float32)


STATE_SIZE = len(TRAINED_FEATURE_ORDER_15DIM)
print("STATE_SIZE =", STATE_SIZE)

# Sanity check against a known real row
_sample_vec = preprocess_row(raw_df.iloc[0])
print("Sample preprocessed vector:", _sample_vec)


## 4. Stratified 5-fold split (BEFORE augmentation)

Splitting first and augmenting each fold's training portion afterwards
matters specifically *because* this is medical data: if you augmented
before splitting, a synthetic row derived from a patient that ends up in
the test set could leak into training (or vice-versa), inflating apparent
accuracy without the model actually generalizing better. Every fold's test
set below stays **100% real, unaugmented data** for the entire notebook.


In [ ]:
N_FOLDS = 5

X_raw = raw_df[CANONICAL_FEATURE_ORDER].reset_index(drop=True)
y_raw = raw_df["target"].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
raw_fold_indices = list(skf.split(X_raw, y_raw))

for i, (train_idx, test_idx) in enumerate(raw_fold_indices):
    print(f"Fold {i+1}: {len(train_idx)} train / {len(test_idx)} test "
          f"(train positive rate {y_raw.iloc[train_idx].mean():.3f}, "
          f"test positive rate {y_raw.iloc[test_idx].mean():.3f})")


## 5. Clinically-safe data augmentation

Synthetic rows are generated **only by interpolating between two real
patients who already share the same diagnosis label**, never by inventing
a feature combination from scratch:

- **Continuous vitals** (`age`, `trestbps`, `thalach`, `oldpeak`): a
  synthetic row's continuous values are a random point on the line between
  a real "anchor" patient and one of its nearest same-class neighbors —
  i.e. a patient that plausibly could exist between two real ones with the
  same outcome. Values are then clipped to the app's own valid clinical
  ranges as a hard safety net.
- **Categorical/binary fields** (`sex`, `fbs`, `exang`, `cp`, `restecg`,
  `slope`): copied *verbatim* from the anchor patient, never blended or
  randomly flipped. This guarantees every synthetic row's categorical
  profile is identical to some real patient's — we only ever nudge a real
  patient's vitals slightly toward a clinically similar peer, we never
  fabricate a new symptom combination.
- Augmentation is applied **separately per class** (not just to rebalance
  the minority class) so both LOW and HIGH risk patients gain diversity.
- `AUGMENTATION_FACTOR = 1.0` below means each fold's *training* set is
  roughly doubled (adds one synthetic row per real training row). Increase
  it for more synthetic diversity, but keep the real:synthetic ratio
  sensible -- this notebook intentionally keeps real data as at least half
  of every training fold.


In [ ]:
AUGMENTATION_FACTOR = 1.0   # synthetic rows added = real rows * this factor
N_NEIGHBORS = 5             # neighborhood size for interpolation, per class


def augment_training_fold(train_df, seed):
    """train_df: real rows only (raw canonical features + target).
    Returns a new dataframe = original real rows + synthetic rows,
    synthetic rows generated by same-class neighbor interpolation.
    """
    rng = np.random.RandomState(seed)
    synthetic_rows = []

    for label in train_df["target"].unique():
        class_df = train_df[train_df["target"] == label].reset_index(drop=True)
        n_synthetic = int(round(len(class_df) * AUGMENTATION_FACTOR))
        if n_synthetic == 0 or len(class_df) < 2:
            continue

        k = min(N_NEIGHBORS, len(class_df) - 1)
        cont = class_df[CONTINUOUS_FEATURES].to_numpy(dtype=np.float64)
        # Standardize just for neighbor-distance purposes (keeps age's
        # larger numeric range from dominating the distance metric) -- this
        # standardization is throwaway, NOT the model's scaler.
        cont_mean, cont_std = cont.mean(axis=0), cont.std(axis=0) + 1e-8
        cont_norm = (cont - cont_mean) / cont_std

        nn_model = NearestNeighbors(n_neighbors=k + 1).fit(cont_norm)
        _, neighbor_idx = nn_model.kneighbors(cont_norm)

        for _ in range(n_synthetic):
            anchor_i = rng.randint(0, len(class_df))
            # neighbor_idx[anchor_i, 0] is the anchor itself; pick another.
            neighbor_choices = neighbor_idx[anchor_i, 1:]
            neighbor_i = neighbor_choices[rng.randint(0, len(neighbor_choices))]

            anchor_row = class_df.iloc[anchor_i]
            neighbor_row = class_df.iloc[neighbor_i]

            synthetic = anchor_row.copy()
            t = rng.uniform(0.0, 1.0)
            for feat in CONTINUOUS_FEATURES:
                interpolated = anchor_row[feat] + t * (neighbor_row[feat] - anchor_row[feat])
                lo, hi = FEATURE_RANGES[feat]
                synthetic[feat] = float(np.clip(interpolated, lo, hi))
            # Categorical features stay exactly as the anchor patient's --
            # see cell markdown above for why.
            synthetic_rows.append(synthetic)

    if not synthetic_rows:
        return train_df.copy()

    synthetic_df = pd.DataFrame(synthetic_rows).reset_index(drop=True)
    augmented = pd.concat([train_df, synthetic_df], ignore_index=True)
    return augmented.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # shuffle


# Build augmented folds: real test set untouched, training set augmented.
folds = []
for i, (train_idx, test_idx) in enumerate(raw_fold_indices):
    train_real = raw_df.iloc[train_idx].reset_index(drop=True)
    test_real = raw_df.iloc[test_idx].reset_index(drop=True)

    train_augmented = augment_training_fold(train_real, seed=SEED + i)

    states_train = np.stack([preprocess_row(r) for _, r in train_augmented.iterrows()])
    labels_train = train_augmented["target"].to_numpy(dtype=np.int64)
    states_test = np.stack([preprocess_row(r) for _, r in test_real.iterrows()])
    labels_test = test_real["target"].to_numpy(dtype=np.int64)

    folds.append({
        "fold": i + 1,
        "states_train": states_train,
        "labels_train": labels_train,
        "states_test": states_test,
        "labels_test": labels_test,
        "state_size": STATE_SIZE,
        "n_real_train": len(train_real),
        "n_synthetic_train": len(train_augmented) - len(train_real),
    })

    print(f"Fold {i+1}: {len(train_real)} real train rows + "
          f"{len(train_augmented) - len(train_real)} synthetic = "
          f"{len(train_augmented)} total train | {len(test_real)} real test rows (untouched)")


## 6. DQN — regularized network + training

Same `DiseasePredictionEnvironment` mechanic as the original notebook
(reward comes from stepping toward the nearest real-or-synthetic patient in
feature space) — it now simply has a much denser, more diverse set of
training states to match against, since `states_train` includes the
augmented rows. Network changes: **dropout** added after each hidden layer,
and **L2 weight decay** added to the optimizer, both specifically to reduce
overfitting on weak-signal features like `age`.


In [ ]:
NUM_EPISODES = 300
MAX_STEPS = 300
GAMMA = 0.99
LEARNING_RATE = 0.001
EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
BATCH_SIZE = 64
MEMORY_SIZE = 10000
TARGET_UPDATE = 10
DQN_WEIGHT_DECAY = 1e-4   # NEW -- L2 regularization to fight overfitting
DQN_DROPOUT = 0.2         # NEW

NUM_FEATURES = STATE_SIZE
ACTION_SIZE = NUM_FEATURES * 3
ACTION_INCREASE, ACTION_DECREASE, ACTION_HOLD = 1, -1, 0
DELTA = 0.05


class DiseasePredictionEnvironment:
    def __init__(self, states, labels):
        self.original_states = np.array(states, dtype=np.float32)
        self.labels = np.array(labels, dtype=np.int64)
        self.num_samples = len(self.original_states)
        self.reset()

    def reset(self):
        self.current_index = np.random.randint(0, self.num_samples)
        self.current_state = self.original_states[self.current_index].copy()
        self.steps = 0
        return self.current_state.copy()

    def modify_state(self, state, action):
        new_state = state.copy()
        feature, operation = action // 3, action % 3
        if operation == ACTION_INCREASE:
            new_state[feature] += DELTA
        elif operation == ACTION_DECREASE:
            new_state[feature] -= DELTA
        return new_state

    def nearest_patient(self, modified_state):
        from sklearn.metrics.pairwise import cosine_similarity
        similarities = cosine_similarity(modified_state.reshape(1, -1), self.original_states)[0]
        best_index = np.argmax(similarities)
        return best_index, similarities[best_index]

    def step(self, action):
        self.steps += 1
        modified_state = self.modify_state(self.current_state, action)
        next_index, similarity = self.nearest_patient(modified_state)
        next_state = self.original_states[next_index].copy()
        label = self.labels[next_index]

        if label == 1:
            reward, done = 10.0, True
        else:
            reward, done = -1.0, False
        if self.steps >= MAX_STEPS:
            done = True

        self.current_index, self.current_state = next_index, next_state.copy()
        return next_state, reward, done, {"label": int(label)}

    def sample_action(self):
        return random.randint(0, ACTION_SIZE - 1)


class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((np.array(state, dtype=np.float32), int(action), float(reward),
                             np.array(next_state, dtype=np.float32), bool(done)))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.FloatTensor(np.array(states)).to(DEVICE),
                torch.LongTensor(np.array(actions)).unsqueeze(1).to(DEVICE),
                torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(DEVICE),
                torch.FloatTensor(np.array(next_states)).to(DEVICE),
                torch.FloatTensor(np.array(dones)).unsqueeze(1).to(DEVICE))

    def __len__(self):
        return len(self.memory)

    def ready(self):
        return len(self.memory) >= BATCH_SIZE


class DeepQNetwork(nn.Module):
    """Same shape as the deployed model (15->128->256->128->ACTION_SIZE),
    with dropout added after each ReLU (NEW)."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(DQN_DROPOUT),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(DQN_DROPOUT),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(DQN_DROPOUT),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.model(x)


class DQNAgent:
    def __init__(self):
        self.policy_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(DEVICE)
        self.target_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(DEVICE)
        self.target_network.load_state_dict(self.policy_network.state_dict())
        self.target_network.eval()
        # weight_decay is the L2 regularization fix (NEW).
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=LEARNING_RATE,
                                     weight_decay=DQN_WEIGHT_DECAY)
        self.criterion = nn.SmoothL1Loss()
        self.memory = ReplayMemory(MEMORY_SIZE)
        self.epsilon = EPSILON_START

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, ACTION_SIZE - 1)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            q_values = self.policy_network(state_t)
        return torch.argmax(q_values).item()

    def remember(self, *args):
        self.memory.push(*args)

    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return None
        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)
        current_q = self.policy_network(states).gather(1, actions)
        with torch.no_grad():
            next_q = self.target_network(next_states).max(1)[0].unsqueeze(1)
            target_q = rewards + (1 - dones) * GAMMA * next_q
        loss = self.criterion(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_network.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def update_target(self):
        self.target_network.load_state_dict(self.policy_network.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon * EPSILON_DECAY, EPSILON_MIN)


def train_one_episode(agent, env):
    state, done, episode_reward, steps = env.reset(), False, 0.0, 0
    while not done and steps < MAX_STEPS:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        agent.replay()
        state = next_state
        episode_reward += reward
        steps += 1
    agent.update_target()
    agent.decay_epsilon()
    return episode_reward


def train_dqn_fold(fold_data):
    env = DiseasePredictionEnvironment(fold_data["states_train"], fold_data["labels_train"])
    agent = DQNAgent()
    for episode in range(NUM_EPISODES):
        reward = train_one_episode(agent, env)
        if (episode + 1) % TARGET_UPDATE == 0:
            agent.update_target()
        if (episode + 1) % 50 == 0:
            print(f"  Fold {fold_data['fold']} | episode {episode+1}/{NUM_EPISODES} | reward {reward:.1f} | eps {agent.epsilon:.3f}")
    return agent


print("=" * 70)
print("TRAINING DQN (5 folds, augmented training data)")
print("=" * 70)
dqn_agents = []
for fold_data in folds:
    print(f"\nFold {fold_data['fold']}")
    agent = train_dqn_fold(fold_data)
    dqn_agents.append(agent)
print("\nDQN training complete.")


## 7. PPO — entropy-stabilized network + training

Same `ActorCritic` architecture as the deployed model
(`shared(15→128→128) + actor(128→128→2, softmax) + critic(128→128→1)`),
plus dropout in the shared/actor layers, and `ENTROPY_COEFFICIENT` raised
from `0.01` to `0.05` (still a constant throughout training, just strong
enough to meaningfully resist the policy collapsing to near-deterministic
0%/100% outputs). The environment now **shuffles the training rows every
episode** (the original notebook always started at index 0 and walked the
same fixed order every episode, which no longer makes sense once there are
augmented rows to see in varied order too).


In [ ]:
PPO_NUM_EPISODES = 500
PPO_LEARNING_RATE = 3e-4
PPO_GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPSILON = 0.2
ENTROPY_COEFFICIENT = 0.05   # CHANGED from 0.01 -- stronger, sustained entropy pressure
VALUE_LOSS_COEFFICIENT = 0.5
PPO_EPOCHS = 10
PPO_BATCH_SIZE = 64
ADAM_EPSILON = 1e-5
MAX_GRAD_NORM = 0.5
HIDDEN_DIM = 128
PPO_ACTION_SIZE = 2
PPO_DROPOUT = 0.15   # NEW


class HeartDiseaseEnvironment:
    """Same reward structure as the original (correct classification = +1,
    incorrect = -1), but now shuffles training row order every episode
    (NEW -- the original notebook always walked states_train starting at
    index 0 in the same fixed order every single episode)."""
    def __init__(self, fold_data, seed):
        self.states_train = fold_data["states_train"]
        self.labels_train = fold_data["labels_train"]
        self.states_test = fold_data["states_test"]
        self.labels_test = fold_data["labels_test"]
        self.rng = np.random.RandomState(seed)
        self.order = None
        self.pos = 0

    def reset(self):
        self.order = self.rng.permutation(len(self.states_train))
        self.pos = 0
        return self.states_train[self.order[self.pos]]

    def step(self, action):
        true_label = self.labels_train[self.order[self.pos]]
        reward = 1 if action == true_label else -1
        self.pos += 1
        done = self.pos >= len(self.order)
        next_state = self.states_train[self.order[self.pos]] if not done else self.states_train[self.order[-1]]
        return next_state, reward, done, {}


class ActorCritic(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(), nn.Dropout(PPO_DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(PPO_DROPOUT),
        )
        self.actor = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(PPO_DROPOUT),
            nn.Linear(HIDDEN_DIM, PPO_ACTION_SIZE), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        self.to(DEVICE)

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)
        state = state.to(DEVICE)
        features = self.shared(state)
        return self.actor(features), self.critic(features)

    def act(self, state):
        action_probs, value = self.forward(state)
        dist = Categorical(action_probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action).detach(), value.detach()

    def evaluate(self, states, actions):
        action_probs, values = self.forward(states)
        dist = Categorical(action_probs)
        return dist.log_prob(actions), values.squeeze(), dist.entropy()


class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self):
        self.states, self.actions, self.rewards = [], [], []
        self.dones, self.log_probs, self.values = [], [], []

    def store(self, state, action, reward, done, log_prob, value):
        self.states.append(state); self.actions.append(action)
        self.rewards.append(reward); self.dones.append(done)
        self.log_probs.append(log_prob); self.values.append(value)

    def __len__(self):
        return len(self.states)

    def compute_returns_and_advantages(self):
        rewards, dones, values = self.rewards, self.dones, [v.item() for v in self.values]
        returns, advantages = [], []
        gae, next_value = 0.0, 0.0
        for t in reversed(range(len(rewards))):
            mask = 1.0 - float(dones[t])
            delta = rewards[t] + PPO_GAMMA * next_value * mask - values[t]
            gae = delta + PPO_GAMMA * GAE_LAMBDA * mask * gae
            advantages.insert(0, gae)
            next_value = values[t]
        returns = [a + v for a, v in zip(advantages, values)]
        self.returns = torch.FloatTensor(returns).to(DEVICE)
        self.advantages = torch.FloatTensor(advantages).to(DEVICE)
        self.states_t = torch.FloatTensor(np.array(self.states)).to(DEVICE)
        self.actions_t = torch.LongTensor(self.actions).to(DEVICE)
        self.log_probs_t = torch.FloatTensor(self.log_probs).to(DEVICE)

    def normalize_advantages(self):
        self.advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)

    def generate_batches(self):
        n = len(self.states)
        indices = np.random.permutation(n)
        for start in range(0, n, PPO_BATCH_SIZE):
            batch_idx = indices[start:start + PPO_BATCH_SIZE]
            yield (self.states_t[batch_idx], self.actions_t[batch_idx],
                   self.log_probs_t[batch_idx], self.returns[batch_idx], self.advantages[batch_idx])


class PPOAgent:
    def __init__(self, state_size):
        self.policy = ActorCritic(state_size).to(DEVICE)
        self.old_policy = ActorCritic(state_size).to(DEVICE)
        self.old_policy.load_state_dict(self.policy.state_dict())
        self.buffer = RolloutBuffer()
        self.optimizer = optim.Adam(self.policy.parameters(), lr=PPO_LEARNING_RATE, eps=ADAM_EPSILON)
        self.mse_loss = nn.MSELoss()

    def select_action(self, state):
        with torch.no_grad():
            return self.old_policy.act(state)

    def store_transition(self, **kwargs):
        self.buffer.store(**kwargs)

    def update_policy(self):
        if len(self.buffer) == 0:
            return
        self.buffer.compute_returns_and_advantages()
        self.buffer.normalize_advantages()

        for _ in range(PPO_EPOCHS):
            for states, actions, old_log_probs, returns, advantages in self.buffer.generate_batches():
                new_log_probs, state_values, entropy = self.policy.evaluate(states, actions)
                ratio = torch.exp(new_log_probs - old_log_probs)
                clipped_ratio = torch.clamp(ratio, 1 - CLIP_EPSILON, 1 + CLIP_EPSILON)
                policy_loss = -torch.min(ratio * advantages, clipped_ratio * advantages).mean()
                value_loss = self.mse_loss(state_values, returns)
                entropy_bonus = entropy.mean()
                # ENTROPY_COEFFICIENT is the key fix -- see cell markdown above.
                total_loss = policy_loss + VALUE_LOSS_COEFFICIENT * value_loss - ENTROPY_COEFFICIENT * entropy_bonus

                self.optimizer.zero_grad()
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), MAX_GRAD_NORM)
                self.optimizer.step()

        self.old_policy.load_state_dict(self.policy.state_dict())
        self.buffer.clear()


def train_ppo_episode(env, agent):
    state, done, episode_reward = env.reset(), False, 0
    while not done:
        action, log_prob, value = agent.select_action(state)
        next_state, reward, done, _ = env.step(action)
        agent.store_transition(state=state, action=action, reward=reward, done=done,
                                log_prob=log_prob, value=value)
        state = next_state
        episode_reward += reward
    agent.update_policy()
    return episode_reward


def train_ppo_fold(fold_data, seed):
    env = HeartDiseaseEnvironment(fold_data, seed=seed)
    agent = PPOAgent(STATE_SIZE)
    agent.policy.train()
    for episode in range(PPO_NUM_EPISODES):
        reward = train_ppo_episode(env, agent)
        if (episode + 1) % 100 == 0:
            print(f"  Fold {fold_data['fold']} | episode {episode+1}/{PPO_NUM_EPISODES} | reward {reward}")
    return agent


print("=" * 70)
print("TRAINING PPO (5 folds, augmented training data)")
print("=" * 70)
ppo_agents = []
for i, fold_data in enumerate(folds):
    print(f"\nFold {fold_data['fold']}")
    agent = train_ppo_fold(fold_data, seed=SEED + i)
    ppo_agents.append(agent)
print("\nPPO training complete.")


## 8. Evaluate both retrained models

Evaluated on each fold's **untouched, 100% real** test set (never on
synthetic rows). Compare against the previously known baselines noted in
this X-REACH codebase's model metadata: PPO ~0.78 accuracy / ~0.85 ROC-AUC,
DQN ~0.50 accuracy / ~0.44 ROC-AUC (both never independently verified for
the specific exported folds actually in use).


In [ ]:
def evaluate_dqn(agent, fold_data):
    agent.policy_network.eval()
    X_test, y_test = fold_data["states_test"], fold_data["labels_test"]
    preds, probs = [], []
    with torch.no_grad():
        for state in X_test:
            q = agent.policy_network(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
            p = torch.softmax(q[:, :2], dim=1)
            preds.append(torch.argmax(q[:, :2], dim=1).item())
            probs.append(p[0, 1].item())
    agent.policy_network.train()
    return preds, probs, y_test


def evaluate_ppo(agent, fold_data):
    agent.policy.eval()
    X_test, y_test = fold_data["states_test"], fold_data["labels_test"]
    preds, probs = [], []
    with torch.no_grad():
        for state in X_test:
            action_probs, _ = agent.policy(torch.FloatTensor(state).unsqueeze(0).to(DEVICE))
            preds.append(torch.argmax(action_probs, dim=1).item())
            probs.append(action_probs[0, 1].item())
    agent.policy.train()
    return preds, probs, y_test


def summarize(name, agents, eval_fn):
    rows = []
    for agent, fold_data in zip(agents, folds):
        preds, probs, y_test = eval_fn(agent, fold_data)
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec = recall_score(y_test, preds, zero_division=0)
        f1 = f1_score(y_test, preds, zero_division=0)
        try:
            auc = roc_auc_score(y_test, probs)
        except ValueError:
            auc = float("nan")
        rows.append({"Fold": fold_data["fold"], "Accuracy": acc, "Precision": prec,
                     "Recall": rec, "F1": f1, "ROC_AUC": auc})
    df = pd.DataFrame(rows)
    print(f"\n=== {name} ===")
    print(df.to_string(index=False))
    print(df.mean(numeric_only=True))
    return df


dqn_results = summarize("DQN", dqn_agents, evaluate_dqn)
ppo_results = summarize("PPO", ppo_agents, evaluate_ppo)


## 9. Sanity check — single-feature monotonicity sweep

The same kind of check used earlier to diagnose the original models'
problems. Run this on the best fold of each retrained model and look for:
(a) PPO confidence values that are no longer pinned to exactly 0%/100%, and
(b) `age` increasing risk roughly monotonically instead of oscillating.


In [ ]:
best_dqn_fold_idx = int(dqn_results["Accuracy"].idxmax())
best_ppo_fold_idx = int(ppo_results["Accuracy"].idxmax())
best_dqn_agent = dqn_agents[best_dqn_fold_idx]
best_ppo_agent = ppo_agents[best_ppo_fold_idx]
print("Best DQN fold:", dqn_results.iloc[best_dqn_fold_idx]["Fold"])
print("Best PPO fold:", ppo_results.iloc[best_ppo_fold_idx]["Fold"])


def predict_proba(model_type, raw_row):
    vec = preprocess_row(raw_row)
    tensor = torch.FloatTensor(vec).unsqueeze(0).to(DEVICE)
    if model_type == "dqn":
        best_dqn_agent.policy_network.eval()
        with torch.no_grad():
            q = best_dqn_agent.policy_network(tensor)
            prob = torch.softmax(q[:, :2], dim=1)[0, 1].item()
        best_dqn_agent.policy_network.train()
    else:
        best_ppo_agent.policy.eval()
        with torch.no_grad():
            action_probs, _ = best_ppo_agent.policy(tensor)
            prob = action_probs[0, 1].item()
        best_ppo_agent.policy.train()
    return prob


base = pd.Series({"thalach": 140, "restecg": 1, "oldpeak": 1.5, "slope": 1, "age": 52,
                   "sex": 1, "cp": 2, "exang": 0, "trestbps": 140, "fbs": 0})

for model_type in ["dqn", "ppo"]:
    print(f"\n--- {model_type.upper()} sweeps (base = moderate-risk profile) ---")
    print("age sweep:")
    for age in [20, 35, 50, 65, 80]:
        row = base.copy(); row["age"] = age
        print(f"  age={age:3d}  risk={predict_proba(model_type, row)*100:5.1f}%")
    print("oldpeak sweep:")
    for oldpeak in [0, 1, 2, 3, 4, 5, 6]:
        row = base.copy(); row["oldpeak"] = oldpeak
        print(f"  oldpeak={oldpeak}  risk={predict_proba(model_type, row)*100:5.1f}%")
    print("thalach sweep (lower = clinically worse):")
    for thalach in [200, 170, 140, 110, 80]:
        row = base.copy(); row["thalach"] = thalach
        print(f"  thalach={thalach:3d}  risk={predict_proba(model_type, row)*100:5.1f}%")


## 10. Export best fold — DQN

Matches the backend's artifact contract exactly (`models/README.md`), with
the `output_activation` fix (`"none"` — DQN's output stays raw Q-values,
softmax is applied by the backend).


In [ ]:
BEST_DQN_DIR = "best_model_dqn"
os.makedirs(BEST_DQN_DIR, exist_ok=True)

best_dqn_fold_number = int(dqn_results.iloc[best_dqn_fold_idx]["Fold"])
best_dqn_row = dqn_results.iloc[best_dqn_fold_idx]

dqn_pth_path = os.path.join(BEST_DQN_DIR, "model.pth")
torch.save(best_dqn_agent.policy_network.state_dict(), dqn_pth_path)

dqn_metadata = {
    "model_version": f"dqn-fold{best_dqn_fold_number}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "DQN",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": "Trained on the 15-dimension representation produced by backend/services/preprocessingService.js. Reuses the deployed models/rl/preprocessing_params.json scaler unchanged.",
        "trained_state_dimension": STATE_SIZE,
    },
    "training_metadata": {
        "fold": best_dqn_fold_number,
        "num_folds": N_FOLDS,
        "episodes": NUM_EPISODES,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    },
    "augmentation_metadata": {
        "method": "same-class nearest-neighbor interpolation on continuous features; categorical features copied verbatim from anchor patient; clipped to clinical validity ranges",
        "augmentation_factor": AUGMENTATION_FACTOR,
        "n_real_train": folds[best_dqn_fold_idx]["n_real_train"],
        "n_synthetic_train": folds[best_dqn_fold_idx]["n_synthetic_train"],
        "test_set": "100% real, unaugmented",
    },
    "evaluation_metadata": {
        "accuracy": float(best_dqn_row["Accuracy"]),
        "precision": float(best_dqn_row["Precision"]),
        "recall": float(best_dqn_row["Recall"]),
        "f1": float(best_dqn_row["F1"]),
        "roc_auc": None if pd.isna(best_dqn_row["ROC_AUC"]) else float(best_dqn_row["ROC_AUC"]),
    },
    "runtime_compatibility": {
        "framework": "pytorch",
        "min_python": "3.10",
        "export_format": "onnx",
        "input_name": "input",
        "num_classes": 2,
        "output_activation": "none",   # DQN outputs raw Q-values
    },
}

dqn_onnx_path = os.path.join(BEST_DQN_DIR, "model.onnx")
best_dqn_agent.policy_network.eval()
torch.onnx.export(
    best_dqn_agent.policy_network,
    torch.randn(1, STATE_SIZE, device=DEVICE),
    dqn_onnx_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
)

dqn_metadata_path = os.path.join(BEST_DQN_DIR, "metadata.json")
with open(dqn_metadata_path, "w") as f:
    json.dump(dqn_metadata, f, indent=2)

print("Saved:", dqn_pth_path, dqn_onnx_path, dqn_metadata_path)

try:
    from google.colab import files
    files.download(dqn_pth_path)
    files.download(dqn_onnx_path)
    files.download(dqn_metadata_path)
except ImportError:
    print("Not in Colab -- files are available locally under", BEST_DQN_DIR)


## 11. Export best fold — PPO

Same fix set as DQN's export, plus the two PPO-specific bugs found in the
original notebook: `output_activation: "softmax"` (PPO's actor head already
applies its own softmax — the backend must NOT apply it again, or you get
exactly the 0%/100% saturation bug this retraining is meant to fix), and
the correct state dimension (15, not the stale hardcoded `10` found in the
original export cell).


In [ ]:
BEST_PPO_DIR = "best_model_ppo"
os.makedirs(BEST_PPO_DIR, exist_ok=True)

best_ppo_fold_number = int(ppo_results.iloc[best_ppo_fold_idx]["Fold"])
best_ppo_row = ppo_results.iloc[best_ppo_fold_idx]

ppo_pth_path = os.path.join(BEST_PPO_DIR, "model.pth")
torch.save({
    "actor_state_dict": best_ppo_agent.policy.state_dict(),
    "critic_state_dict": best_ppo_agent.policy.critic.state_dict(),
}, ppo_pth_path)

ppo_metadata = {
    "model_version": f"ppo-fold{best_ppo_fold_number}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "PPO",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": "Trained on the 15-dimension representation produced by backend/services/preprocessingService.js. Architecture: shared(15->128->128) + actor(128->128->2, softmax) + critic(128->128->1), with dropout added (see Section 7).",
        "trained_state_dimension": STATE_SIZE,
    },
    "training_metadata": {
        "fold": best_ppo_fold_number,
        "num_folds": N_FOLDS,
        "episodes": PPO_NUM_EPISODES,
        "entropy_coefficient": ENTROPY_COEFFICIENT,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    },
    "augmentation_metadata": {
        "method": "same-class nearest-neighbor interpolation on continuous features; categorical features copied verbatim from anchor patient; clipped to clinical validity ranges",
        "augmentation_factor": AUGMENTATION_FACTOR,
        "n_real_train": folds[best_ppo_fold_idx]["n_real_train"],
        "n_synthetic_train": folds[best_ppo_fold_idx]["n_synthetic_train"],
        "test_set": "100% real, unaugmented",
    },
    "evaluation_metadata": {
        "accuracy": float(best_ppo_row["Accuracy"]),
        "precision": float(best_ppo_row["Precision"]),
        "recall": float(best_ppo_row["Recall"]),
        "f1": float(best_ppo_row["F1"]),
        "roc_auc": None if pd.isna(best_ppo_row["ROC_AUC"]) else float(best_ppo_row["ROC_AUC"]),
    },
    "runtime_compatibility": {
        "framework": "pytorch",
        "min_python": "3.10",
        "export_format": "onnx",
        "input_name": "input",
        "num_classes": 2,
        "output_activation": "softmax",   # PPO's actor head already applies softmax -- CRITICAL, see markdown above
    },
}


class ActorOnly(nn.Module):
    """Wraps shared+actor for a clean single-output ONNX graph (critic not needed for inference)."""
    def __init__(self, actor_critic):
        super().__init__()
        self.shared = actor_critic.shared
        self.actor = actor_critic.actor

    def forward(self, state):
        return self.actor(self.shared(state))


best_ppo_agent.policy.eval()
actor_only = ActorOnly(best_ppo_agent.policy).eval()

ppo_onnx_path = os.path.join(BEST_PPO_DIR, "model.onnx")
torch.onnx.export(
    actor_only,
    torch.randn(1, STATE_SIZE, device=DEVICE),   # CORRECT dimension (15) -- fixes the original notebook's stale hardcoded 10
    ppo_onnx_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=13,
)

ppo_metadata_path = os.path.join(BEST_PPO_DIR, "metadata.json")
with open(ppo_metadata_path, "w") as f:
    json.dump(ppo_metadata, f, indent=2)

print("Saved:", ppo_pth_path, ppo_onnx_path, ppo_metadata_path)

try:
    from google.colab import files
    files.download(ppo_pth_path)
    files.download(ppo_onnx_path)
    files.download(ppo_metadata_path)
except ImportError:
    print("Not in Colab -- files are available locally under", BEST_PPO_DIR)


## 12. Deploying the retrained models

1. In the X-REACH repo, create two new folders:
   - `models/rl/dqn-fold<N>-<date>/`
   - `models/rl/ppo-fold<N>-<date>/`

   (use the exact `model_version` string printed in each `metadata.json`).
2. Copy that fold's `model.pth`, `model.onnx`, and `metadata.json` into the
   matching folder.
3. **No backend code changes and no database reseeding are needed** — the
   scaler parameters are unchanged from what's already deployed.
4. Restart the backend (`node --watch` picks up the new files automatically
   on the next prediction request since `modelService.js` discovers models
   at process start — a restart is still needed since discovery only runs
   once at startup).
5. Re-run this backend's model comparison / sanity checks (or trust
   Section 9 above) before treating this as more reliable than the
   currently deployed artifacts — retraining with augmentation and
   stronger regularization is a genuine attempt at a fix, not a
   guaranteed one. Validate against real held-out clinical data before any
   real diagnostic use.
